# Common Statistical Tests as Linear Models
This notebook explores the thesis from Jonas Kristoffer Lindeløv: that many common statistical tests are essentially linear models under the hood. We will demonstrate this using Python's `scipy.stats` for traditional tests and `statsmodels` for linear models.

In [3]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as smf

# Set seed for reproducibility
np.random.seed(42)

## 1. Student's t-test as a Linear Model
The independent t-test compares the means of two groups. In a linear model context, this is equivalent to predicting the outcome variable `y` using a binary categorical variable for the groups.

In [4]:
# Generate simulated data
group_A = np.random.normal(loc=10, scale=2, size=50)
group_B = np.random.normal(loc=12, scale=2, size=50)

df_t = pd.DataFrame({
    'y': np.concatenate([group_A, group_B]),
    'group': ['A']*50 + ['B']*50
})

# 1. Traditional t-test
t_stat, t_pvalue = stats.ttest_ind(group_A, group_B, equal_var=True)
print(f"Traditional t-test p-value: {t_pvalue:.5f}")

# 2. Linear Model equivalent (y ~ group)
# statsmodels automatically treats strings as categorical dummy variables
lm_t = smf.ols('y ~ group', data=df_t).fit()
print(f"Linear Model p-value for group difference: {lm_t.pvalues['group[T.B]']:.5f}")

Traditional t-test p-value: 0.00000
Linear Model p-value for group difference: 0.00000


## 2. Mann-Whitney U Test as a Linear Model
The Mann-Whitney test is non-parametric, meaning it compares the *ranks* of the data rather than the raw values. Therefore, the linear model equivalent is simply an ordinary least squares (OLS) regression run on the ranked data.

In [5]:
# Generate non-normal data (exponential distribution)
group_A_non_normal = np.random.exponential(scale=1/0.2, size=50)
group_B_non_normal = np.random.exponential(scale=1/0.5, size=50)

df_mw = pd.DataFrame({
    'y': np.concatenate([group_A_non_normal, group_B_non_normal]),
    'group': ['A']*50 + ['B']*50
})

# 1. Traditional Mann-Whitney U test
mw_stat, mw_pvalue = stats.mannwhitneyu(group_A_non_normal, group_B_non_normal, alternative='two-sided')
print(f"Traditional Mann-Whitney p-value: {mw_pvalue:.5f}")

# 2. Linear Model on the Ranks
df_mw['ranked_y'] = stats.rankdata(df_mw['y'])
lm_mw = smf.ols('ranked_y ~ group', data=df_mw).fit()
print(f"Linear Model on ranks p-value: {lm_mw.pvalues['group[T.B]']:.5f}")
# Note: P-values are nearly identical. Slight variances occur because scipy applies specific continuity corrections.

Traditional Mann-Whitney p-value: 0.00552
Linear Model on ranks p-value: 0.00490


## 3. Wilcoxon Signed-Rank Test as a Linear Model
This test is the non-parametric equivalent of a paired t-test. It looks at the difference between two paired groups, signs them, and ranks them. The linear model equivalent is an intercept-only model (predicting against 1) on the signed ranks.

In [6]:
# Generate paired data
pre_test = np.random.normal(loc=50, scale=10, size=30)
post_test = pre_test + np.random.normal(loc=5, scale=5, size=30)

# 1. Traditional Wilcoxon signed-rank test
wilcox_stat, wilcox_pvalue = stats.wilcoxon(post_test, pre_test, alternative='two-sided')
print(f"Traditional Wilcoxon p-value: {wilcox_pvalue:.5f}")

# 2. Linear Model equivalent
differences = post_test - pre_test
# Calculate signed ranks
ranks = stats.rankdata(np.abs(differences))
signed_ranks = np.sign(differences) * ranks

df_wilcox = pd.DataFrame({'signed_ranks': signed_ranks})

# Intercept-only linear model (y ~ 1)
lm_wilcox = smf.ols('signed_ranks ~ 1', data=df_wilcox).fit()
print(f"Linear Model (intercept only) p-value: {lm_wilcox.pvalues['Intercept']:.5f}")

Traditional Wilcoxon p-value: 0.00000
Linear Model (intercept only) p-value: 0.00000


## 4. The Wald Test in Linear Models
The Wald test tests whether a parameter is significantly different from a hypothesized value (usually 0). When we look at a summary table for a linear regression, the standard t-statistic and p-value calculated for our slopes *are* Wald tests. The statistic is simply: `Estimate / Standard Error`.

In [7]:
# Let's extract the values from our first linear model (lm_t)
estimate = lm_t.params['group[T.B]']
std_error = lm_t.bse['group[T.B]']

# 1. Calculate the Wald t-statistic manually
manual_wald_t = estimate / std_error
print(f"Manually calculated Wald t-statistic: {manual_wald_t:.5f}")

# 2. Extract the t-statistic calculated by statsmodels
statsmodels_t = lm_t.tvalues['group[T.B]']
print(f"Statsmodels calculated t-statistic: {statsmodels_t:.5f}")

Manually calculated Wald t-statistic: 6.87273
Statsmodels calculated t-statistic: 6.87273


## Conclusion
By understanding the General Linear Model, we can replicate the behavior of Student's t-test, the Mann-Whitney U test, and the Wilcoxon signed-rank test simply by manipulating our input data (e.g., using dummy coding or ranking) and evaluating it via OLS regression. The significance of these model parameters is inherently evaluated using the Wald test.